In [1]:
import plotly.io as pio
pio.renderers.default = "notebook_connected" # Render inline inside the IDE


In [2]:
import sys
!{sys.executable} -m pip install plotly


'c:\Users\Acer\Desktop\GOOGLE_PLAY_STORE_ANALYSIS' is not recognized as an internal or external command,
operable program or batch file.


## Let's import some Pyhthon Libraries to make our Project work in real world.

In [3]:
import pandas as pd # Data manipulation library
import numpy as np # Numerical operations library
import plotly.express as px # Plotting library
import random # Random Library

In [4]:
# dataset = pd.read_csv("D:/GOOGLE_PLAY_STORE_ANALYSIS/Play Store Data.csv") # Load dataset
# dataset
dataset = pd.read_csv("../Play Store Data.csv")


In [5]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  str    
 1   Category        10841 non-null  str    
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  str    
 4   Size            10841 non-null  str    
 5   Installs        10841 non-null  str    
 6   Type            10840 non-null  str    
 7   Price           10841 non-null  str    
 8   Content Rating  10840 non-null  str    
 9   Genres          10841 non-null  str    
 10  Last Updated    10841 non-null  str    
 11  Current Ver     10833 non-null  str    
 12  Android Ver     10838 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.1 MB


In [6]:
dataset.describe()

,Rating
count,9367.000000
mean,4.193338
std,0.537431
min,1.000000
25%,4.000000
50%,4.300000
75%,4.500000
max,19.000000


In [7]:
dataset.isnull().sum() # Ratings has so many missing values, so I will drop all of them.

App                  0
Category             0
Rating            1474
Reviews              0
Size                 0
Installs             0
Type                 1
Price                0
Content Rating       1
Genres               0
Last Updated         0
Current Ver          8
Android Ver          3
dtype: int64

In [8]:
dataset

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10836,Sya9a Maroc - FR,FAMILY,4.5,38,53M,"5,000+",Free,0,Everyone,Education,"July 25, 2017",1.48,4.1 and up
10837,Fr. Mike Schmitz Audio Teachings,FAMILY,5.0,4,3.6M,100+,Free,0,Everyone,Education,"July 6, 2018",1.0,4.1 and up
10838,Parkinson Exercices FR,MEDICAL,NaN,3,9.5M,"1,000+",Free,0,Everyone,Medical,"January 20, 2017",1.0,2.2 and up
10839,The SCP Foundation DB fr nn5n,BOOKS_AND_REFERENCE,4.5,114,Varies with device,"1,000+",Free,0,Mature 17+,Books & Reference,"January 19, 2015",Varies with device,Varies with device


In [9]:
dataset = dataset.dropna(subset=["Rating", "Content Rating", "Type"], inplace = False) # Drop rows where "Rating" or "Content Rating" are missing
dataset.isnull().sum() # Null values are gone.

App               0
Category          0
Rating            0
Reviews           0
Size              0
Installs          0
Type              0
Price             0
Content Rating    0
Genres            0
Last Updated      0
Current Ver       4
Android Ver       2
dtype: int64

## So, let's start with the Task.

In [10]:
dataset = dataset.copy() # Create a copy of the dataset to avoid modifying the original one
dataset.loc[:, 'Reviews'] = pd.to_numeric(dataset['Reviews'], errors='coerce')
data = dataset[dataset['Reviews'] > 1000] # Filter dataset to include only rows where 'Reviews' is greater than 1000
print(data) # Print the shape of the filtered dataset

TypeError: Invalid value '[   159    967  87510 ...      4    114 398307]' for dtype 'str'

In [ ]:
rating_bins = [0, 2, 4, 5]  # Apps rated from 0-2, 2-4, and 4-5 stars
rating_labels = ['1-2 Stars', '3-4 Stars', '4-5 Stars']  # These labels will show in the final chart

# Data Preparation for Sentiment and Rating Analysis

#### I filter apps with more than 1,000 reviews, categorize them into rating groups, and assign random sentiment labels. The following visualizations will illustrate sentiment distribution across app categories and rating groups.

In [ ]:
data = data.copy()
data.loc[:, 'rating_group'] = pd.cut(
    data['Rating'],
    bins=rating_bins,
    labels=rating_labels,
    include_lowest=True
)

In [ ]:
print(data[['App', 'Rating', 'rating_group']].sample(5)) # It will print random rows from the dataset with 'App', 'Rating', and 'rating_group' columns

In [ ]:
grouped = data.groupby(
    ['rating_group', 'Type'], 
    observed=True
).size().reset_index(name='App Count')
print(grouped)

# I will make a Sentiment Distribution Bar Graph to provide a quick visual summary.

In [ ]:
data = data.copy()

if 'Sentiment' not in data.columns:
    sentiments = ['Positive', 'Neutral', 'Negative']
    data.loc[:, 'Sentiment'] = [random.choice(sentiments) for _ in range(len(data))]

fig = px.bar(
    data,
    x='Category',
    y='Reviews',
    color='Sentiment',
    barmode='stack',
    facet_col='rating_group',
    title='Sentiment Distribution by App Category and Rating Group',
    color_discrete_map={
        'Positive': 'green',
        'Neutral': 'orange',
        'Negative': 'red'
    }
)
fig.update_layout(width=1000, height=500)
fig.show()


# I want to make a Pie Graph

In [11]:
sentiment_counts = data['Sentiment'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

# Create pie chart
fig = px.pie(
    sentiment_counts,
    names='Sentiment',
    values='Count',
    title='Overall Sentiment Distribution',
    color='Sentiment',
    color_discrete_map={
        'Positive': 'green',
        'Neutral': 'yellow',
        'Negative': 'red'
    }
)

fig.update_traces(textinfo='percent+label')
fig.update_layout(width=600, height=600)
fig.show()

NameError: name 'data' is not defined